# V4 segmentation versus official LIDC masks

使用 **roivue** 比较 V4 最终分割与官方标注：整幅 CT 的轴向、冠状、矢状和 3D 视图，以及逐结节的联动放大视图。
**红色 = V4 prediction；蓝色 = Official consensus。** 勾选图层、调整透明度，可分别查看两个 mask。

选择 **`visual` kernel**，然后按顺序运行。默认使用 V4 **epoch 18** 和测试集病例 **LIDC-IDRI-0301__scan301**。
本目录缓存首次推理结果；再次打开只需 CPU。新病例会使用已分配的 GPU，或自动提交一个单卡 Slurm 作业。
在参数单元格修改 `CASE_ID` 可切换病例，设为 `None` 则按 `SEED` 抽取一个测试病例。

这里的官方参考是各医生 mask 按物理结节合并得到的 **consensus**：至少两位医生同意，只有一位医生时使用其 mask。
包括 malignancy 均值为 3 的结节。推理本身不使用官方 mask 或 GT 框。

文件保留了实际运行的指标结果；交互视图需运行对应单元格加载，也可直接打开最后一节导出的 HTML。

## 1. 配置

`visual` 环境已安装本地 roivue。若换了 kernel，可在该 kernel 中运行
`%pip install -e /home/users/yj255/code/imaging_feature/roivue`，并确保安装了 pandas、ipykernel。
模型推理由独立的 `pro6000` 环境执行，显示 kernel 不需要 PyTorch。

In [ ]:
from pathlib import Path
import sys
import random
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "back_prop/evaluate/eval_package").is_dir()),
            Path("/home/users/yj255/code/imaging_feature"))
NOTEBOOK_DIR = ROOT / "back_prop/evaluate/visualize_mask"
ROIVUE_REPO = ROOT / "roivue"
for path in (ROOT, ROIVUE_REPO):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

CHECKPOINT = Path("/usr/project/rudinlab/datasets/LIDC_IDRI/joint_model/model_v4_runs/12748637/epoch_018.pt")
MANIFEST = ROOT / "vista3D/data/folds/fold_0.json"
INFERENCE_PYTHON = Path("/home/users/yj255/.local/miniforge3/envs/pro6000/bin/python")
RESULT_ROOT = NOTEBOOK_DIR / "results/v4_epoch018"

CASE_ID = "LIDC-IDRI-0301__scan301"  # None: random test case with the fixed seed below
SEED = 4
FORCE_INFERENCE = False
EXECUTION = "auto"                  # auto / local / slurm
PARTITION, ACCOUNT, GRES = "rudin", "rudin", "gpu:a6000:1"
MASK_THRESHOLD = 0.5
MATCH_IOU = 0.1
WINDOW_HU = (-1000, 400)            # lower and upper HU, not center/width
MIN_ROI_MM = 60.
SCAN_MAX_VOXELS = 12_000_000        # overview only; close-ups/metrics use full resolution
EXPORT_HTML = True                 # write a standalone overview and the initial selected ROI

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import ipywidgets as W
from IPython.display import display, Markdown, FileLink
from roivue import view as cv, export
from back_prop.evaluate.eval_package import load_result, load_test_cases, nodule_segmentation
from back_prop.evaluate.eval_package.annotations import case_key
from back_prop.evaluate.eval_package.runner import ensure_case_result
from back_prop.evaluate.visualize_mask.roivue_helpers import (
    whole_scan_case, nodule_case, selection_label, close_widgets,
)

cases = sorted(load_test_cases(MANIFEST, split="testing"), key=case_key)
case_ids = [case_key(c) for c in cases]
selected_case_id = CASE_ID if CASE_ID is not None else random.Random(SEED).choice(case_ids)
if selected_case_id not in case_ids:
    raise ValueError(f"Not in the testing split: {selected_case_id}. Available IDs: {case_ids}")
print(f"Kernel: {sys.executable}")
print(f"Case: {selected_case_id} | test cases: {len(case_ids)}")
print(f"Checkpoint: {CHECKPOINT}")
# To list every available test case: display(pd.Series(case_ids, name="case_id"))

## 2. 读取缓存或运行 V4

首次新病例需要等待推理完成，作业号及日志目录会显示在下面。中断等待不会取消 GPU 作业；重跑会继续等待同一个作业。
缓存保存 CT、affine、最终预测和官方 consensus。所有显示和指标使用该共同网格，避免把原始 CT 与重采样后的 mask 混用。

In [ ]:
result_dir = ensure_case_result(
    checkpoint=CHECKPOINT, manifest=MANIFEST, case_id=selected_case_id,
    output_root=RESULT_ROOT, python=INFERENCE_PYTHON, root=ROOT,
    force=FORCE_INFERENCE, execution=EXECUTION,
    partition=PARTITION, account=ACCOUNT, gres=GRES,
    mask_threshold=MASK_THRESHOLD, minimum_iou=MATCH_IOU,
)
result = load_result(result_dir)
assert result.metadata["architecture"] == "whole_ct_radiomics_offset_groupnorm_early_diagnostics_v4"
assert result.metadata["split"] == "testing"
assert result.metadata["ground_truth_used_for_prediction"] is False
assert result.metadata["bank_unchanged"] is True
rows = nodule_segmentation(result)
print(f"Loaded {result.case_id}: {len(result.ground_truth)} official nodules, "
      f"{len(result.predictions)} selected model queries")
print(f"Final masks: probability >= {result.metadata['mask_threshold']}; "
      f"query selection: object probability >= {result.metadata['object_threshold']}")
print(f"Grid: {result.shape}, axes: {nib.aff2axcodes(result.affine)}, "
      f"spacing (mm): {np.round(nib.affines.voxel_sizes(result.affine), 4)}")
print(f"Official reader masks: {result.metadata['official_mask_dir']}")

## 3. 定量对比

整幅 CT 的指标比较所有最终预测 mask 的并集与所有官方 consensus 的并集。
逐结节表按 IoU ≥ `MATCH_IOU` 做一对一匹配；`missed_gt` 为未匹配的官方结节，
`unmatched_prediction` 为未匹配的预测。空预测也保留在表中；两侧均空时 Dice/IoU 的约定值为 1。
`nodule_id` 是官方物理结节编号，`query_id` 是模型 query 编号，两者并非同一编号体系。

In [ ]:
scan_case, scan_metrics = whole_scan_case(result, window=WINDOW_HU)
voxel_mm3 = float(abs(np.linalg.det(result.affine[:3, :3])))
summary = {
    "case_id": result.case_id,
    "Dice": scan_metrics["dice"], "IoU": scan_metrics["iou"],
    "V4 volume (mm3)": scan_metrics["predicted_voxels"] * voxel_mm3,
    "Official volume (mm3)": scan_metrics["gt_voxels"] * voxel_mm3,
    **{name: sum(row["status"] == name for row in rows)
       for name in ("matched", "missed_gt", "unmatched_prediction")},
}
scan_table = pd.DataFrame([summary])
display(scan_table.round(4))

records = []
for i, row in enumerate(rows):
    gt = None if row["gt_index"] is None else result.ground_truth[row["gt_index"]]
    pred = None if row["prediction_index"] is None else result.predictions[row["prediction_index"]]
    records.append({
        "ROI": i, "status": row["status"], "nodule_id": row["nodule_id"],
        "query_id": row["query_id"], "Dice": row["metrics"]["dice"],
        "IoU": row["metrics"]["iou"],
        "V4 voxels": row["metrics"]["predicted_voxels"],
        "Official voxels": row["metrics"]["gt_voxels"],
        "object probability": None if pred is None else pred.object_probability,
        "reader annotation IDs": "" if gt is None else ", ".join(map(str, gt.annotation_ids)),
    })
nodule_table = pd.DataFrame(records, columns=[
    "ROI", "status", "nodule_id", "query_id", "Dice", "IoU", "V4 voxels", "Official voxels",
    "object probability", "reader annotation IDs",
])
for col in ("ROI", "nodule_id", "query_id"):
    nodule_table[col] = nodule_table[col].astype("Int64")
display(nodule_table.round(4))

## 4. 整幅 CT：V4 与官方 mask 叠加

拖动或滚轮浏览切片，在 3D 区域拖动旋转。用下方复选框分别开关红色预测和蓝色官方 mask，
`opacity` 调整透明度。为控制浏览器内存，整幅 CT 视图可能降采样，`info` 可查看显示分辨率。
小结节请在下一节以完整分辨率检查；指标始终在完整网格计算。

In [ ]:
close_widgets(globals().get("scan_widget"))
scan_view = cv.load(scan_case, scan_max_voxels=SCAN_MAX_VOXELS)
scan_widget = scan_view.scan(height=430, opacity=0.55)
display(scan_widget)

## 5. 逐结节：原始 CT、mask 和 3D 表面

下拉菜单与上表的 ROI 编号对应。八个视图联动，包含原始 CT、叠加 mask 的三视图及 3D 表面。
每个条目只显示所选 query 与对应官方结节的 mask。放大范围覆盖**该配对中预测与官方 mask 的并集**，因此超出官方边界的预测仍可见。
未匹配的结节只显示存在的一侧；没有非零体素的条目会给出提示。
3D 表面由 roivue 平滑显示，Dice/IoU 使用未平滑的二值 mask。

In [ ]:
close_widgets(globals().get("nodule_widget"))
close_widgets(globals().get("roi_controls"))
nodule_widget = None
nodule_view = None
roi_picker = W.Dropdown(
    options=[(selection_label(i, row), i) for i, row in enumerate(rows)],
    description="Nodule:", layout=W.Layout(width="95%"),
    disabled=not bool(rows),
)
roi_output = W.Output()

def show_nodule(change=None):
    global nodule_widget, nodule_view
    close_widgets(nodule_widget)
    nodule_widget, nodule_view = None, None
    roi_output.clear_output(wait=True)
    with roi_output:
        if roi_picker.value is None:
            print("No official nodules or model queries in this scan.")
            return
        row = rows[roi_picker.value]
        display(Markdown(f"**{selection_label(roi_picker.value, row)}**"))
        case = nodule_case(result, row, window=WINDOW_HU, min_mm=MIN_ROI_MM)
        if case is None:
            print("This entry contains no nonzero mask voxels on the CT grid; no 3D surface is available.")
            return
        if case.meta["absent_masks"]:
            print("Absent mask:", ", ".join(case.meta["absent_masks"]))
        nodule_view = cv.load(case)
        nodule_widget = nodule_view.region(0, tile_height=240, min_mm=MIN_ROI_MM, opacity=0.6)
        display(nodule_widget)

roi_picker.observe(show_nodule, names="value")
roi_controls = W.VBox([roi_picker, roi_output])
display(roi_controls)
show_nodule()

## 6. 保存 HTML 和指标

CSV 和 HTML 保存到当前病例的 `roivue/` 子目录。HTML 可下载后直接在浏览器打开，不需要 Python。
首次导出会下载并缓存 NiiVue JavaScript；随后 HTML 自包含。`EXPORT_HTML=False` 可跳过自动导出。
切换上方结节后，点击按钮可导出当前结节。

In [ ]:
export_dir = result_dir / "roivue"
export_dir.mkdir(parents=True, exist_ok=True)
scan_table.to_csv(export_dir / "scan_metrics.csv", index=False)
nodule_table.to_csv(export_dir / "nodule_metrics.csv", index=False)
(export_dir / "provenance.json").write_text(json.dumps({
    "case_id": result.case_id, "metadata": result.metadata,
    "match_iou": result.minimum_iou, "window_hu": WINDOW_HU,
    "scan_max_voxels": SCAN_MAX_VOXELS,
    "color_legend": {"V4 prediction": "red", "Official consensus": "blue"},
}, indent=2) + "\n")

def save_selected_html():
    if nodule_view is None:
        print("No nonempty ROI selected.")
        return None
    path = export.write_html(
        nodule_view, export_dir / f"nodule_{roi_picker.value:03d}.html",
        title=f"{result.case_id} | {selection_label(roi_picker.value, rows[roi_picker.value])}",
        regions=1, min_mm=MIN_ROI_MM,
    )
    print(path)
    display(FileLink(str(path.relative_to(NOTEBOOK_DIR))))
    return path

if EXPORT_HTML:
    overview_html = export.write_html(
        scan_view, export_dir / "whole_scan.html",
        title=f"{result.case_id} | V4 (red) vs official consensus (blue)", regions=0,
    )
    print(overview_html)
    display(FileLink(str(overview_html.relative_to(NOTEBOOK_DIR))))
    save_selected_html()

close_widgets(globals().get("export_controls"))
export_button = W.Button(description="Export selected nodule", icon="save", layout=W.Layout(width="230px"))
export_output = W.Output()
def export_selected(_):
    with export_output:
        export_output.clear_output(wait=True)
        save_selected_html()
export_button.on_click(export_selected)
export_controls = W.VBox([export_button, export_output])
display(export_controls)
print(f"CSV / HTML directory: {export_dir}")